# Serving a trained LoRA adapter with vLLM

Takes the adapter a training run kept (for example the
[LoRA Distributed Training Test](lora-distributed-3gpu.ipynb)) and serves it on SUSE AI Factory:

1. reads the adapter's `adapter_config.json` from the run's checkpoint volume: its base model and rank;
2. starts a vLLM endpoint for that base model with LoRA on, as an AI Factory workload (a Blueprint of
   the vLLM chart from the SUSE Application Collection, and an AIWorkload from it);
3. copies the adapter onto the endpoint's model volume, in the cluster (`volumes.copy`: the files never
   leave it, and every file's checksum is compared);
4. loads it into the running server, without a restart, and asks the base model and the adapter the
   same question.

**Needs:** the SDK with `volumes.copy` (the install cell below); AI Factory's Application Collection
credentials (Settings); a GPU for the base model (one whole GPU, or a KAI share of one); and, to
create the Blueprint (a cluster-wide object), a user who may create Blueprints, such as an
administrator. The endpoint and the run must be in the same project: a pod mounts only volumes of its
own namespace.

The notebook talks to vLLM through the Kubernetes API's service proxy, so it runs from a laptop as
well as from inside the cluster.

## 1. The SDK

In [ ]:
%pip install -q "rancher-ai[pandas] @ https://github.com/dstanley/aif/archive/refs/heads/pr/7-designs.tar.gz#subdirectory=sdk/python"

## 2. Settings

`RUN` is the training run whose adapter to serve; its checkpoint volume is `<RUN>-checkpoints` and
the adapter is in `<RUN>/adapter` on it. `ADAPTER` is the name requests will use for it.

In [ ]:
CONTEXT = "gb200"                  # kubeconfig context of the cluster
PROJECT = "ai-data-science"        # the project's namespace: the run's, and the endpoint's
RUN = "train-aobpo"                # the training run whose adapter to serve
ADAPTER = "suse-32b"               # the model name requests use for the adapter
ENDPOINT = "lora-serve"            # the AI Factory workload, and the vLLM release

GPUS = 1                           # whole GPUs for the server (a 32B model in bf16 needs about 65 GB)
GPU_SHARE_MIB = 0                  # or a KAI share of one GPU instead, in MiB (with GPUS = 0)
MODEL_STORAGE = "100Gi"            # the model's volume: the base weights, and the adapters
STORAGE_CLASS = ""                 # "" = the cluster's default
SERVER_CPU = 4                     # the server pod's CPU request
SERVER_MEMORY = "32Gi"             # and its memory request
MAX_MODEL_LEN = 8192
OVERWRITE = False                  # replace an adapter of the same name already on the volume
RECREATE = False                   # redeploy an existing endpoint with section 4's settings as they are now

## 3. The adapter: what it was trained on

`adapter_config.json` names the base model and the rank. The server must serve the same base model,
and vLLM must allow the rank (`--max-lora-rank`, 16 by default).

In [ ]:
import json, tempfile, time
from rancher_ai import Client

ai = Client(context=CONTEXT, project=PROJECT)
CHECKPOINTS = f"{RUN}-checkpoints"
with tempfile.TemporaryDirectory() as d:
    cfg = json.load(open(ai.checkpoints.get(CHECKPOINTS).get(f"{RUN}/adapter/adapter_config.json", d + "/")))
BASE, RANK = cfg["base_model_name_or_path"], int(cfg["r"])
MAX_RANK = next(n for n in (8, 16, 32, 64, 128, 256, 320, 512) if n >= RANK)
print(f"adapter of {BASE}: rank {RANK}, on {cfg.get('target_modules')}")

## 4. The vLLM endpoint

A Blueprint with one component, the Application Collection's vLLM chart (the one the SUSE
inference blueprints use): the base model on its own volume, LoRA on (`--enable-lora`), room for the
adapter's rank, and `VLLM_ALLOW_RUNTIME_LORA_UPDATING` so adapters can be loaded while it runs. Its
router gives one OpenAI-compatible address in front of it. The AIWorkload then deploys it into the
project, like any AI Factory endpoint.

In [ ]:
AIF = ("ai-factory.suse.com", "v1alpha1")
FAMILY = f"{ENDPOINT}-{PROJECT}"[:50].rstrip("-")      # Blueprints are cluster-wide: one per project and endpoint
VERSION = "1.0.0"
BP_NAME = f"{FAMILY}-{VERSION.replace('.', '-')}"
MODEL = "base"                                          # the chart's name for the model entry
VOLUME = f"{ENDPOINT}-{MODEL}-storage-claim"            # the chart names the model's volume so
ENGINE = f"{ENDPOINT}-{MODEL}-engine-service"           # ... and the server's Service
ROUTER = f"{ENDPOINT}-router-service"
# the project's KAI queue, as AI Factory's Projects page labels the namespace: the server's GPU counts
# against the project's quota (a pod without it lands in KAI's default queue, which has none)
QUEUE = (ai.conn.core.read_namespace(PROJECT).metadata.labels or {}).get("kai.scheduler/queue")
if not QUEUE:
    raise RuntimeError(f"{PROJECT} has no kai.scheduler/queue label: is it an AI Factory project with a GPU quota?")

engine = {
    "name": MODEL, "modelURL": BASE, "replicaCount": 1,
    "registry": "dp.apps.rancher.io", "repository": "containers/vllm-openai", "tag": "0.19.0-5.29",
    "imagePullPolicy": "IfNotPresent",
    "requestCPU": SERVER_CPU, "requestMemory": SERVER_MEMORY, "requestGPU": GPUS,
    "pvcStorage": MODEL_STORAGE, "pvcAccessMode": ["ReadWriteOnce"], **({"storageClass": STORAGE_CLASS} if STORAGE_CLASS else {}),
    "vllmConfig": {
        "maxModelLen": MAX_MODEL_LEN,
        "extraArgs": ["--enable-lora", "--max-lora-rank", str(MAX_RANK), "--max-loras", "4",
                      "--gpu-memory-utilization", "0.85" if GPU_SHARE_MIB else "0.9"],
    },
    "env": [{"name": "VLLM_ALLOW_RUNTIME_LORA_UPDATING", "value": "True"}],
}
if GPU_SHARE_MIB:
    engine.update(requestGPU=0, podAnnotations={"gpu-memory": str(GPU_SHARE_MIB)})

values = {
    "servingEngineSpec": {
        "enableEngine": True, "schedulerName": "kai-scheduler", "labels": {"kai.scheduler/queue": QUEUE},
        "runtimeClassName": "" if GPU_SHARE_MIB else "nvidia",
        "containerPort": 8000, "servicePort": 80, "modelSpec": [engine],
    },
    "routerSpec": {
        "enableRouter": True, "replicaCount": 1, "routingLogic": "roundrobin", "serviceDiscovery": "k8s",
        "k8sServiceDiscoveryType": "pod-ip", "serviceType": "ClusterIP", "servicePort": 80, "containerPort": 8000,
        "registry": "dp.apps.rancher.io", "repository": "containers/lmcache-lmstack-router", "tag": "0.1.10-3.60",
        "imagePullPolicy": "IfNotPresent",
        "resources": {"requests": {"cpu": "400m", "memory": "1000Mi"}, "limits": {"memory": "1000Mi"}},
        "ingress": {"enabled": False},
    },
}
blueprint = {
    "apiVersion": "/".join(AIF), "kind": "Blueprint",
    "metadata": {"name": BP_NAME, "labels": {"ai-factory.suse.com/blueprint-name": FAMILY,
                                             "ai-factory.suse.com/blueprint-version": VERSION}},
    "spec": {"displayName": f"vLLM with LoRA: {BASE.split('/')[-1]} ({PROJECT})", "source": "Custom", "version": VERSION,
             "category": "inference",
             "description": f"{BASE} on vLLM with LoRA adapters loadable at run time, from {PROJECT}'s serve-adapter notebook.",
             "components": [{"chartRepo": "application-collection", "chartName": "vllm", "chartVersion": "0.1.10",
                             "releaseName": ENDPOINT, "values": values}]},
}

custom = ai.conn.custom
try:
    cur = custom.get_cluster_custom_object(*AIF, "blueprints", BP_NAME)
    blueprint["metadata"]["resourceVersion"] = cur["metadata"]["resourceVersion"]
    custom.replace_cluster_custom_object(*AIF, "blueprints", BP_NAME, blueprint)
    print("updated Blueprint", BP_NAME)
except Exception as e:
    if getattr(e, "status", None) != 404:
        raise
    custom.create_cluster_custom_object(*AIF, "blueprints", blueprint)
    print("created Blueprint", BP_NAME)
print("queue", QUEUE)

In [ ]:
workload = {
    "apiVersion": "/".join(AIF), "kind": "AIWorkload",
    "metadata": {"name": ENDPOINT, "namespace": PROJECT},
    "spec": {"displayName": f"vLLM + LoRA: {ADAPTER}", "deployStrategy": "FleetBundle",
             "source": {"sourceType": "Blueprint", "blueprint": {"name": FAMILY, "version": VERSION}},
             "targetClusters": [ai.cluster_id], "targetNamespace": PROJECT},
}
# AI Factory renders the Blueprint when the workload is created: a changed Blueprint reaches an
# existing endpoint only by deploying it again (RECREATE), which restarts the server
try:
    custom.get_namespaced_custom_object(*AIF, PROJECT, "aiworkloads", ENDPOINT)
    exists = True
except Exception as e:
    if getattr(e, "status", None) != 404:
        raise
    exists = False
if exists and RECREATE:
    custom.delete_namespaced_custom_object(*AIF, PROJECT, "aiworkloads", ENDPOINT)
    while True:
        try:
            custom.get_namespaced_custom_object(*AIF, PROJECT, "aiworkloads", ENDPOINT)
        except Exception:
            break
        time.sleep(5)
    while ai.conn.core.list_namespaced_pod(PROJECT, label_selector=f"helm-release-name={ENDPOINT}").items:
        time.sleep(5)
    print("removed the old endpoint")
if exists and not RECREATE:
    print(f"AIWorkload {ENDPOINT} already exists: kept as it is (RECREATE = True deploys it again with section 4's settings)")
else:
    custom.create_namespaced_custom_object(*AIF, PROJECT, "aiworkloads", workload)
    print("created AIWorkload", ENDPOINT)

Wait for the server. The first start downloads the base model to its volume (65 GB for a 32B model:
10 to 30 minutes), then loads it onto the GPU.

In [ ]:
core = ai.conn.core
selector = f"helm-release-name={ENDPOINT},model={MODEL}"
start, last = time.time(), ""
while True:
    pods = core.list_namespaced_pod(PROJECT, label_selector=selector).items
    p = pods[0] if pods else None
    ready = bool(p and p.status.container_statuses and all(c.ready for c in p.status.container_statuses))
    if ready:
        print(f"server ready on {p.spec.node_name} after {(time.time() - start) / 60:.0f} min")
        break
    if p:
        waiting = [c.state.waiting.reason for c in (p.status.container_statuses or []) if c.state.waiting]
        state = f"{p.status.phase} {' '.join(waiting)}".strip()
    else:
        wl = custom.get_namespaced_custom_object(*AIF, PROJECT, "aiworkloads", ENDPOINT).get("status", {})
        state = f"no pod yet (workload {wl.get('phase', 'Pending')}: " + "; ".join(
            c.get("message", "") for c in wl.get("conditions", []) if c.get("status") == "False")[:200] + ")"
    if state != last:
        print(f"{(time.time() - start) / 60:4.0f} min  {state}")
        last = state
    if time.time() - start > 90 * 60:
        raise TimeoutError("the server is not ready after 90 minutes: see its pod's events and log")
    time.sleep(15)

## 5. Copy the adapter onto the model's volume

In the cluster: a short Job mounts the run's checkpoint volume and the server's model volume, copies
the adapter folder to `lora-adapters/<ADAPTER>`, compares every file's SHA-256, and only then puts
it in place. It runs on the server's node, where its volume is mounted.

In [ ]:
copied = ai.volumes.copy(CHECKPOINTS, f"{RUN}/adapter", VOLUME, f"lora-adapters/{ADAPTER}", overwrite=OVERWRITE)
copied

## 6. Load it into the server

vLLM's `/v1/load_lora_adapter` loads it by path, while the server runs. The notebook reaches the
server's Service through the Kubernetes API (`services/proxy`), so nothing needs exposing.

In [ ]:
def vllm(method, path, body=None, service=ENGINE):
    # a call to the vLLM server (or its router) through the Kubernetes API's service proxy, with the
    # kubeconfig's own credentials and TLS (calls that every Kubernetes client release has)
    api = ai.conn.core.api_client
    cfg = api.configuration
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    headers.update({a["key"]: a["value"] for a in cfg.auth_settings().values() if a.get("in") == "header" and a.get("value")})
    r = api.rest_client.request(method, f"{cfg.host}/api/v1/namespaces/{PROJECT}/services/{service}:80/proxy{path}",
                                headers=headers, body=body, _request_timeout=600)
    data = r.read() if callable(getattr(r, "read", None)) else None
    text = (data if data is not None else r.data).decode()
    return json.loads(text) if text.strip().startswith(("{", "[")) else text

loaded = [m["id"] for m in vllm("GET", "/v1/models")["data"]]
if ADAPTER in loaded and OVERWRITE:
    vllm("POST", "/v1/unload_lora_adapter", {"lora_name": ADAPTER})
if ADAPTER not in loaded or OVERWRITE:
    print(vllm("POST", "/v1/load_lora_adapter", {"lora_name": ADAPTER, "lora_path": f"/data/lora-adapters/{ADAPTER}"}))
print("models:", [m["id"] for m in vllm("GET", "/v1/models")["data"]])

## 7. Ask the base model and the adapter

The same question to both, with greedy decoding, so any difference comes from the adapter.

In [ ]:
QUESTION = "How do I add the Helm repository for SUSE Rancher Prime?"

def ask(model, question=QUESTION, max_tokens=200):
    r = vllm("POST", "/v1/chat/completions", {"model": model, "messages": [{"role": "user", "content": question}],
                                              "max_tokens": max_tokens, "temperature": 0})
    return r["choices"][0]["message"]["content"]

for model in (BASE, ADAPTER):
    print(f"--- {model}\n{ask(model)}\n")

## 8. Using it

Inside the cluster, clients send OpenAI-style requests to the server's Service,
`http://<ENDPOINT>-base-engine-service.<PROJECT>.svc/v1`, with `"model": "<ADAPTER>"` for the adapter
or the base model's name for the base model. Use the server's Service, not the chart's router
(`<ENDPOINT>-router-service`): the router routes only base models (a request naming the adapter gets
"model not found"), and it finds every vLLM server in the namespace, other endpoints' too.

A loaded adapter does not survive a server restart: after one, run section 6 again. To have it
loaded at every start, add `"--lora-modules", f"{ADAPTER}=/data/lora-adapters/{ADAPTER}"` to the
engine's `extraArgs` in section 4 and run section 4 again with `RECREATE = True`; the adapter stays on
the volume, so section 5 is not needed again.

A newer adapter: train, then run sections 3, 5 and 6 again with `OVERWRITE = True` (the same name),
or with a new `ADAPTER` name (served side by side, up to `--max-loras`).

In [ ]:
vllm("GET", "/v1/models")

## 9. Removing it (optional)

Deleting the AIWorkload uninstalls the endpoint, and the Blueprint goes with the same cell. The
model's volume is deleted with the endpoint, the copied adapters on it too (the base model downloads
again on the next deploy); the adapter itself stays on the run's checkpoint volume.

In [ ]:
REMOVE = False
if REMOVE:
    custom.delete_namespaced_custom_object(*AIF, PROJECT, "aiworkloads", ENDPOINT)
    custom.delete_cluster_custom_object(*AIF, "blueprints", BP_NAME)
    print("removed", ENDPOINT, "and", BP_NAME)